In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer

In [ ]:
train_df = pd.read_csv("crime_train.csv")
test_df = pd.read_csv("crime_test.csv")

train_df = train_df.drop_duplicates()
test_df = test_df.drop_duplicates()

n_train = len(train_df)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

In [ ]:
# Combine for cleaning
df = pd.concat([train_df, test_df], ignore_index=True)

# Remove first three columns
df = df.iloc[:, 3:]

# Separate target variable (Yes = 1, No = 0)
y = df["closed"].map({"Yes": 1, "No": 0})

# Separate features
X = df.drop(columns=["closed"])

# Impute missing weapon values with mode (fit on TRAIN rows only)
imputer = SimpleImputer(strategy="most_frequent")
imputer.fit(X.iloc[:n_train][["weapon"]])
X[["weapon"]] = imputer.transform(X[["weapon"]])

# One hot encode text columns
X = pd.get_dummies(X, dtype=float)

X.head()

In [ ]:
X_np = X.values.astype(float)
y_np = y.values.astype(float)

X_train, X_test = X_np[:n_train], X_np[n_train:]
y_train, y_test = y_np[:n_train], y_np[n_train:]

X_mean = np.mean(X_train, axis=0)
X_std = np.std(X_train, axis=0)

X_train_scaled = (X_train - X_mean) / X_std
X_test_scaled = (X_test - X_mean) / X_std

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def gradient_descent(X, y, learning_rate=0.1,
                     tolerance=1e-6, max_iterations=10000):

    m = len(y)
    n = X.shape[1]

    w = np.zeros(n)
    b = 0

    previous_cost = float('inf')

    for i in range(max_iterations):

        y_pred = sigmoid(X.dot(w) + b)

        error = y_pred - y

        cost = -(1 / m) * np.sum(y * np.log(y_pred + 1e-9) + (1 - y) * np.log(1 - y_pred + 1e-9))

        dw = (1 / m) * X.T.dot(error)
        db = (1 / m) * np.sum(error)

        w = w - learning_rate * dw
        b = b - learning_rate * db

        if abs(previous_cost - cost) < tolerance:
            break

        previous_cost = cost

    return w, b, i + 1

In [ ]:
w, b, iterations = gradient_descent(X_train_scaled, y_train)

print("Weights:", w)
print("Bias:", b)
print("Iterations:", iterations)

In [ ]:
test_pred = (sigmoid(X_test_scaled.dot(w) + b) >= 0.5).astype(int)
accuracy = np.mean(test_pred == y_test)

print("Test Accuracy:", accuracy * 100, "%")